In [16]:
import requests
import time
import pandas as pd
from datetime import datetime

def get_contest_data_to_dataframe(contest_id,participants_limit):
    # Obtener información del concurso
    contest_url = f"https://codeforces.com/api/contest.standings?contestId={contest_id}&from=1&count={participants_limit}&showUnofficial=false"
    contest_response = requests.get(contest_url)
    contest_data = contest_response.json()
    
    if contest_data['status'] != 'OK':
        print("Error al obtener datos del concurso")
        return None
    
    contest_info = contest_data['result']['contest']
    contest_name = contest_info['name']
    contest_start_time = datetime.fromtimestamp(contest_info['startTimeSeconds']).strftime('%Y-%m-%d')
    
    # Obtener participantes
    standings_url = f"https://codeforces.com/api/contest.standings?contestId={contest_id}&from=1&count={participants_limit}&showUnofficial=false"
    standings_response = requests.get(standings_url)
    standings_data = standings_response.json()
    
    if standings_data['status'] != 'OK':
        print("Error al obtener standings")
        return None
    
    problems = standings_data['result']['problems']
    rows = standings_data['result']['rows']
    num_problems = len(problems)
    
    # Obtener todos los handles
    handles = [row['party']['members'][0]['handle'] for row in rows]
    
    # Obtener metadatos de usuarios
    users_info_url = f"https://codeforces.com/api/user.info?handles={';'.join(handles)}"
    users_response = requests.get(users_info_url)
    users_data = users_response.json().get('result', []) if users_response.json()['status'] == 'OK' else []
    
    # Mapear metadatos
    user_metadata = {user['handle']: {
        'country': user.get('country'),
        'city': user.get('city'),
        'rating': user.get('rating'),
        'maxRating': user.get('maxRating')
    } for user in users_data}

    participants_data = []
    
    for row in rows:
        handle = row['party']['members'][0]['handle']
        participant_data = {
            "author_handle": handle,
            "contest_name": contest_name,
            "contest_start_time": contest_start_time,
            **user_metadata.get(handle, {
                'country': None,
                'city': None,
                'rating': None,
                'maxRating': None
            })
        }
        
        # Procesar problemas
        for i in range(num_problems):
            problem_index = chr(65 + i)
            problem_result = row['problemResults'][i]
            
            # Datos básicos
            participant_data.update({
                f"finished_{problem_index}": problem_result['points'] > 0,
                f"rating_{problem_index}": problems[i].get('rating')
            })
            
            # Inicializar campos de envío
            participant_data[f"language_{problem_index}"] = None
            participant_data[f"relative_time_{problem_index}"] = None
            participant_data[f"time_to_answer_{problem_index}"] = None
        
        # Obtener envíos
        status_url = f"https://codeforces.com/api/contest.status?contestId={contest_id}&handle={handle}"
        status_response = requests.get(status_url)
        time.sleep(1)
        
        if status_response.json()['status'] == 'OK':
            submissions = status_response.json()['result']
            prev_time = 0
            
            for submission in submissions:
                if submission['verdict'] == 'OK':
                    problem_index = submission['problem']['index']
                    participant_data[f"language_{problem_index}"] = submission['programmingLanguage']
                    rt = submission['relativeTimeSeconds']
                    participant_data[f"relative_time_{problem_index}"] = rt
                    participant_data[f"time_to_answer_{problem_index}"] = rt - prev_time
                    prev_time = rt
        
        participants_data.append(participant_data)
    
    # Crear DataFrame
    df = pd.DataFrame(participants_data)
    
    # Ordenar columnas
    ordenadito = ['author_handle']
    
    
    for prefix in ['finished', 'language', 'relative_time', 'time_to_answer', 'rating']:
        ordenadito.extend([f"{prefix}_{chr(65 + i)}" for i in range(num_problems)]) # A, B, C, ... el char 65 es A
    
    ordenadito = ordenadito +[
        'contest_name', 'contest_start_time', 
        'country', 'city', 'rating', 'maxRating'
    ]
    return df[ordenadito]  # Retornar el DataFrame con las columnas ordenadas

# Uso
contest_id = 566
participants_limit = 10  # Limitar a los primeros 10 participantes
df = get_contest_data_to_dataframe(contest_id,participants_limit)

if df is not None:
    df.to_csv(f"codeforces_contest_{contest_id}_participants6.csv", index=False)

In [ ]:
#Para obtener el CSV iterar en todos los ID de concurso que quieras

In [ ]:
# Paso 1 filtrar concursos por fecha y nombre de concurso

url = "https://codeforces.com/api/contest.list"
response = requests.get(url)
data= response.json()
data


# Filtrar concursos por fecha y nombre
filtrado = []
if response.status_code == 200:  
    data = response.json()  
    if data['status'] == 'OK':  
        for contest in data['result']['name']:  
            # Verifica si existe 'startTimeSeconds' en el concurso
            if 'startTimeSeconds' in contest:
                start_time = datetime.fromtimestamp(contest['startTimeSeconds'])  
                if datetime(2024, 7, 1) <= start_time <= datetime(2024, 12, 31):  # Límite de fechas
                    filtrado.append({  
                        'id_concurso': contest['id'],  
                        'nombre_concurso': contest['name'],  
                        'hora_inicio_concurso': start_time,  
                        'duracion_(segundos)': contest['durationSeconds'],  
                        'tipo': contest['type'],  
                        })

# Convertir a DataFrame para visualizar mejor 
df = pd.DataFrame(filtrado)
df_filtrado = df[df["nombre_concurso"].str.contains("round|hello|good bye", case=False, na=False)]

lista_de_concursos = df_filtrado['id_concurso'].tolist()  # Obtener lista de IDs de concursos filtrados
#lista_de_concursos.to_csv("lista_de_concursos.csv", index=False)

for lista in lista_de_concursos:
    df = get_contest_data_to_dataframe(lista,participants_limit=10)
    if df is not None:
        df.to_csv(f"codeforces_contest_{lista}_num_participantes_{participants_limit}.csv", index=False)

#Así obtendrás un CSV por cada concurso que cumpla con los criterios de búsqueda, con los primeros 10 participantes de cada concurso., cuyo límite se puede extender.